# 📚 Import libraries

In [1]:
import hashlib
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from collections import defaultdict
import itertools
from threading import Thread
import os # For API keys
import warnings # To suppress warnings
import matplotlib.pyplot as plt
import spacy
from scipy.stats import norm


# Embedding model configuration

In [ ]:
# --- Embedding Model Configuration ---
# Choose your embedding provider and model here
EMBEDDING_CONFIG = {
    # Options: "spacy", "sentence-transformers", "huggingface", "openai"
    "provider": "huggingface",
    #"provider": "sentence-transformers",

    # --- Provider Specific Settings ---
    # spacy: Model name (e.g., "en_core_web_sm", "en_core_web_md", "en_core_web_lg")
    "spacy_model_name": "en_core_web_sm",

    # sentence-transformers: Model name from Hugging Face Hub (e.g., "all-MiniLM-L6-v2", "multi-qa-mpnet-base-dot-v1")
    "st_model_name": "all-MiniLM-L6-v2",

    # huggingface: Model name from Hugging Face Hub (e.g., "bert-base-uncased", "roberta-base")
    "hf_model_name": "bert-base-uncased",
    "hf_pooling": "mean", # Options: "mean", "cls"

    # openai: Model name (e.g., "text-embedding-ada-002", "text-embedding-3-small")
    "openai_model_name": "text-embedding-ada-002",
}

# --- Global variables to hold loaded models (lazy loading) ---
_nlp = None
_st_model = None
_hf_tokenizer = None
_hf_model = None
_openai_client = None

# --- Model Loading and Embedding Function ---

def initialize_embedding_model():
    """Loads the embedding model based on EMBEDDING_CONFIG."""
    global _nlp, _st_model, _hf_tokenizer, _hf_model, _openai_client
    provider = EMBEDDING_CONFIG["provider"]

    if provider == "spacy":
        if _nlp is None:
            model_name = EMBEDDING_CONFIG["spacy_model_name"]
            try:
                _nlp = spacy.load(model_name)
                print(f"Loaded spaCy model: {model_name}")
            except OSError:
                print(f"SpaCy model '{model_name}' not found. Please install it: python -m spacy download {model_name}")
                raise
    elif provider == "sentence-transformers":
        if _st_model is None:
            try:
                from sentence_transformers import SentenceTransformer
                model_name = EMBEDDING_CONFIG["st_model_name"]
                _st_model = SentenceTransformer(model_name)
                print(f"Loaded Sentence Transformer model: {model_name}")
            except ImportError:
                print("Sentence Transformers library not found. Please install it: pip install sentence-transformers")
                raise
    elif provider == "huggingface":
        if _hf_model is None or _hf_tokenizer is None:
            try:
                # Suppress warnings about weights not being used for sequence classification
                warnings.filterwarnings("ignore", category=UserWarning, module='transformers.modeling_utils')
                from transformers import AutoTokenizer, AutoModel
                import torch
                warnings.filterwarnings("default", category=UserWarning, module='transformers.modeling_utils') # Reset warning

                model_name = EMBEDDING_CONFIG["hf_model_name"]
                _hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
                _hf_model = AutoModel.from_pretrained(model_name)
                # Use GPU if available
                if torch.cuda.is_available():
                   _hf_model.to('cuda')
                print(f"Loaded Hugging Face model and tokenizer: {model_name}")
            except ImportError:
                print("Hugging Face Transformers library or PyTorch not found. Please install them: pip install transformers torch")
                raise
    elif provider == "openai":
        if _openai_client is None:
            try:
                from openai import OpenAI
                # Requires OPENAI_API_KEY environment variable to be set
                if not os.getenv("OPENAI_API_KEY"):
                    print("Warning: OPENAI_API_KEY environment variable not set for OpenAI embeddings.")
                    # Optionally raise an error here if you require OpenAI
                    # raise ValueError("OPENAI_API_KEY environment variable not set.")
                _openai_client = OpenAI() # Client automatically picks up the key from env var
                print(f"Initialized OpenAI client for model: {EMBEDDING_CONFIG['openai_model_name']}")
            except ImportError:
                print("OpenAI library not found. Please install it: pip install openai")
                raise
    else:
        raise ValueError(f"Unsupported embedding provider: {provider}")

# Define Classes & Functions

## Classes

In [ ]:
class RagThread(Thread):
    def __init__(self, group=None, target=None, name=None, args=(), kwargs={}, verbose=None):
        # Initializing the Thread class
        super().__init__(group, target, name, args, kwargs)
        self._return = None

    # Overriding the Thread.run function
    def run(self):
        if self._target is not None:
            self._return = self._target(*self._args, **self._kwargs)

    def join(self):
        super().join()
        return self._return

## Functions

In [ ]:


def tokenize_and_filter(text):
    """Splits text into sentences instead of tokens."""
    tokenizer_nlp = es.get_tokenizer_nlp()

    if tokenizer_nlp == 'fallback':
        # Basic fallback if spaCy isn't available
        # Simple sentence splitting by punctuation and cleanup
        import re

        # Split on common sentence endings
        sentences = re.split(r'(?<=[.!?])\s+', text)

        # Remove empty sentences and strip whitespace
        sentences = [sentence.strip() for sentence in sentences if sentence.strip()]

        return sentences

    # Use spaCy for more robust sentence tokenization
    doc = tokenizer_nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    return sentences


#####
def explain_by_token_removal(chunk, query, response):
    """
    Explains the influence of each token in the chunk and query
    by measuring the change in embedding distance of the response
    when each token is removed.
    """
    response_embedding = es.get_embedding(response) # Embedding of the actual RAG response
    query_embedding = es.get_embedding(query)

    chunk_sentences = tokenize_and_filter(chunk)
    query_sentences = tokenize_and_filter(query)

    token_impact = defaultdict(list)

    print("Analyzing chunk tokens wrt response...")
    remove_and_compute(chunk_sentences, response_embedding, "context", token_impact, query)

    print("Analyzing chunk tokens wrt query...")
    remove_and_compute(chunk_sentences, query_embedding, "context", token_impact, query)

    print("Analyzing query tokens wrt response...")
    remove_and_compute(query_sentences, response_embedding, "query", token_impact, chunk)

    for key in  token_impact.keys():
        token_impact[key] = np.mean(token_impact[key])


    return token_impact

#